In [ ]:
import os
import glob
import logging
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from keras.models import load_model

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0' 
logging.getLogger('tensorflow').setLevel(logging.ERROR)

import tensorflow as tf

tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

from keras import layers, models, losses
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

In [ ]:

def load_and_process_all_files(file_list, alpha=0.20):
    X_all, Y_all = [], []
    print(f"Starting loading and EMA Decluttering for {len(file_list)} files...")
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   
        people_xy = data['people_xy']   
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
        mag_reshaped = mag.reshape(T, 1, 120, 18) 
        
        bg = np.copy(mag_reshaped[0])
        decluttered = np.zeros_like(mag_reshaped)
        
        for t in range(T):
            bg = alpha * mag_reshaped[t] + (1 - alpha) * bg
            decluttered[t] = np.abs(mag_reshaped[t] - bg)
        

        flat_coords = people_xy.reshape(T, 8)
        combined_target = np.concatenate([flat_coords, people_mask], axis=1)

        X_all.append(decluttered)
        Y_all.append(combined_target)
        
        print(f"File {i+1}/{len(file_list)} processed. ({T} frame pre-calcolati)")

    X = np.concatenate(X_all, axis=0).astype(np.float32)
    Y = np.concatenate(Y_all, axis=0).astype(np.float32)
    return X, Y


val_indices = [23, 20, 3, 15, 7, 11] 
train_indices = [22, 16, 17, 18, 19, 21, 0, 1, 2, 4, 12, 13, 14, 5, 6, 8, 9, 10]

#tutti_i_file = glob.glob("dataset/data/*.npz")
tutti_i_file = glob.glob("dataset/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]


print("\n--- PREPARING TRAINING SET ---")
X_train, Y_train = load_and_process_all_files(train_files)

print("\n--- PREPARING VALIDATION SET ---")
X_val, Y_val = load_and_process_all_files(val_files)

print("\n==================================================")
print("ALL DATA READY IN RAM!")
print(f"Total individual Train FRAMES: {X_train.shape[0]}")
print(f"Total individual Validation FRAMES: {X_val.shape[0]}")
print("==================================================")

In [ ]:
def masked_mse(y_true, y_pred):
    """
    Calcola l'errore sulle coordinate (MSE). 
    In futuro potremo azzerarlo se la maschera è 0.
    """
    return losses.mean_squared_error(y_true, y_pred)

In [ ]:
def true_masked_mse(y_true_combined, y_pred_coords):
    y_true_coords = y_true_combined[:, :8]
    mask_1d = y_true_combined[:, 8:] 
    mask_2d = tf.repeat(mask_1d, 2, axis=1) 
    
    raw_mse = tf.square(y_true_coords - y_pred_coords)
    
    masked_mse = raw_mse * mask_2d
    
    sum_mse_per_frame = tf.reduce_sum(masked_mse, axis=1)
    valid_elements_per_frame = tf.reduce_sum(mask_2d, axis=1) + 1e-6
    
    return sum_mse_per_frame / valid_elements_per_frame

def true_masked_rmse_metres(y_true_combined, y_pred_coords):
    y_true_coords = y_true_combined[:, :8]
    mask_1d = y_true_combined[:, 8:] 
    mask_2d = tf.repeat(mask_1d, 2, axis=1) 
    
    raw_mse = tf.square(y_true_coords - y_pred_coords)
    masked_mse = raw_mse * mask_2d
    
    sum_mse_per_frame = tf.reduce_sum(masked_mse, axis=1)
    valid_elements_per_frame = tf.reduce_sum(mask_2d, axis=1) + 1e-6
    
    return tf.sqrt(sum_mse_per_frame / valid_elements_per_frame)

In [ ]:
# ARCHITETTURA EEAI-NET V1-Heavy 

def build_eeai_model_v1_heavy(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")

    # BLOCCO 1: Estrazione Base 
    x = layers.Conv2D(32, (1, 5), padding='same', activation='relu', name="conv_1a")(inputs)
    x = layers.MaxPooling2D((1, 2), name="pool_1")(x)
    x = layers.Conv2D(32, (1, 5), padding='same', activation='relu', name="conv_1b")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_2")(x) 
    
    # BLOCCO 2: Livello Intermedio 
    x = layers.SeparableConv2D(64, (1, 3), padding='same', activation='relu', name="sep_conv_1")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_3")(x)
    x = layers.SeparableConv2D(64, (1, 3), padding='same', activation='relu', name="sep_conv_2")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_4")(x) 
    
    # BLOCCO 3: Feature di Alto Livello 
    x = layers.Conv2D(128, (1, 3), padding='same', activation='relu', name="conv_3a")(x)
    
    # Compattazione
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    
    # COLLO DI BOTTIGLIA 
    x = layers.Dense(128, activation='relu', name="features_deep")(x)
    x = layers.Dropout(0.3, name="drop_features")(x) 
    common_feat = layers.Dense(64, activation='relu', name="features")(x)

    # OUTPUT MULTI-HEAD INVARIATI
    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)

    return models.Model(inputs=inputs, outputs=[coords_output, mask_output], name="EEAI_Net_V1_Heavy")


model_heavy = build_eeai_model_v1_heavy()

model_heavy.compile(
    optimizer='adam',
    loss={"coords_head": true_masked_mse, "mask_head": "binary_crossentropy"}, 
    loss_weights={"coords_head": 1.0, "mask_head": 0.5},
    metrics={
        #"coords_head": [tf.keras.metrics.RootMeanSquaredError(name="metres")],
        #"mask_head": ["accuracy"]
        "coords_head": [true_masked_rmse_metres],
        "mask_head": [tf.keras.metrics.BinaryAccuracy(name="bin_acc")]
    }
)

# CHECKPOINT 
checkpoint_heavy = ModelCheckpoint(
    "eeai_best_model_romano_heavy.keras", 
    monitor="val_loss", 
    save_best_only=True, 
    verbose=1
)

# LEARNING RATE SCHEDULER 
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.5,        
    patience=3,        
    min_lr=1e-6,       
    verbose=1
)

# EARLY STOPPING 
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=10,        
    restore_best_weights=True,
    verbose=1
)

EPOCHS = 100 

print("\n--- STARTING TRAINING ---")
history_light = model_heavy.fit(
    train_gen,                
    validation_data=val_gen,  
    epochs=EPOCHS,
    callbacks=[checkpoint_heavy, reduce_lr, early_stop], 
    verbose=1
)
print("--- TRAINING COMPLETED ---")

In [ ]:
def embedded_summary(model, input_shape=(1, 120, 18), is_int8=False):
    total_params = model.count_params()
    
    bytes_per_param = 1 if is_int8 else 4
    estimated_flash_kb = (total_params * bytes_per_param) / 1024
    
    bytes_per_activation = 1 if is_int8 else 4
    max_adjacent_ram_kb = 0
    
    previous_layer_size = (np.prod(input_shape) * bytes_per_activation) / 1024
    
    for layer in model.layers:
        if layer.__class__.__name__ == 'InputLayer' or not hasattr(layer, 'output_shape'):
            continue
            
        output_shape = layer.output_shape
        if isinstance(output_shape, list):
            num_elements = sum([np.prod([dim for dim in shape[1:] if dim is not None]) for shape in output_shape])
        else:
            num_elements = np.prod([dim for dim in output_shape[1:] if dim is not None])
            
        current_layer_size = (num_elements * bytes_per_activation) / 1024
        current_peak = previous_layer_size + current_layer_size
        
        if current_peak > max_adjacent_ram_kb:
            max_adjacent_ram_kb = current_peak
            
        previous_layer_size = current_layer_size

    mode_str = "INT8 (Quantized)" if is_int8 else "FLOAT32 (Training)"

    print("============================================")
    print(f"   HARDWARE REQUIREMENTS REPORT [{mode_str}]   ")
    print("============================================")
    print(f" Estimated FLASH Memory: ~{estimated_flash_kb:.2f} KB (Limit: 800 KB)")
    print(f" Estimated SRAM Memory:  ~{max_adjacent_ram_kb:.2f} KB (Limit: 300 KB)")
    print("============================================\n")

In [ ]:

file_target = "dataset/window_000007.npz"

if not os.path.exists(file_target):
    print(f"ERRORE: Non trovo il file {file_target}")
else:
    data = np.load(file_target)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] 
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    print("Elaborazione filtri e previsioni in corso (V2)...")
    mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2).reshape(T, 1, 120, 18)
    decluttered = np.zeros_like(mag)
    bg = np.copy(mag[0])
    alpha = 0.05
    for t in range(T):
        bg = alpha * mag[t] + (1 - alpha) * bg
        decluttered[t] = np.abs(mag[t] - bg)

    preds = model_heavy.predict(decluttered, verbose=0)
    p_coords = preds[0].reshape(T, 4, 2)
    p_mask = preds[1]
    print("Dati pronti! Inizializzazione Radar...")

    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(9, 11))
            ax.set_xlim(-0.5, 5.3); ax.set_ylim(-0.5, 7.7)
            ax.grid(True, linestyle=':', alpha=0.6)
            ax.set_title(f"Radar V2 | Frame: {frame_idx}/{T-1} | Window: 07", fontsize=14, fontweight='bold')

            stanza = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=3, edgecolor='navy', facecolor='whitesmoke')
            ax.add_patch(stanza)

            for i in range(4):
                is_present = bool(gt_mask[frame_idx, i] > 0.5)
                if is_present:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=250, edgecolors='black', marker='o', label='REALE (GT)' if i==0 else "")
                    ax.text(rx, ry + 0.2, f"P{i+1}", color='darkgreen', fontweight='bold', ha='center')

                conf = float(p_mask[frame_idx, i])
                if conf >= soglia:
                    px, py = p_coords[frame_idx, i]
                    alpha_val = max(0.3, conf)
                    ax.scatter(px, py, c='red', s=200, marker='X', edgecolors='darkred', alpha=alpha_val, label='PREDETTO' if i==0 else "")
                    ax.text(px, py - 0.3, f"{conf*100:.0f}%", color='red', fontsize=10, ha='center', fontweight='bold')

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if by_label:
                ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True, shadow=True)

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    slider_frame = widgets.IntSlider(value=500, min=10, max=T-1, step=1, description='Frame:')
    slider_soglia = widgets.FloatSlider(value=0.50, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    ui = widgets.VBox([slider_frame, slider_soglia, out])
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)